<a href="https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd

REPO_DIR = "/content/flyrank-ml-internship-starter"
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"

# Clone repository if it is not already available
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        check=True
    )

# Load the starter dataset
csv_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

print("Repository exists:", os.path.isdir(REPO_DIR))
print("CSV exists:", os.path.isfile(csv_path))

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

Repository exists: True
CSV exists: True
Dataset loaded successfully.
Shape: (30000, 44)
Rows: 30000
Columns: 44


In [4]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
]

label = [
    "trend_direction",
    "trend_pct",
]

context = [
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "impression_tier",
    "position_tier",
]

excluded = {
    "content_id": "Identifier; useful for traceability, not as a predictive signal.",
    "client_id": "Identifier/group field; should not be used as a model feature.",
    "provider_used": "Generation/provider metadata; not part of the core content-refresh signal.",
    "model_used": "Generation/model metadata; not part of the core content-refresh signal.",
}

print("FEATURES:")
print(features)

print("\nLABEL / OUTCOME:")
print(label)

print("\nCONTEXT:")
print(context)

print("\nEXCLUDED:")
for field, reason in excluded.items():
    print(f"- {field}: {reason}")


FEATURES:
['search_volume', 'competition', 'word_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

LABEL / OUTCOME:
['trend_direction', 'trend_pct']

CONTEXT:
['content_type', 'main_intent', 'age_tier', 'freshness_tier', 'impression_tier', 'position_tier']

EXCLUDED:
- content_id: Identifier; useful for traceability, not as a predictive signal.
- client_id: Identifier/group field; should not be used as a model feature.
- provider_used: Generation/provider metadata; not part of the core content-refresh signal.
- model_used: Generation/model metadata; not part of the core content-refresh signal.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

# Missing values in fields used
fields_used = features + label + context

print("\nMissing values in fields used:")
missing = df[fields_used].isna().sum().sort_values(ascending=False)
print(missing.to_string())

# Window availability
window_cols = [
    "impressions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "sessions_90d",
    "sessions_last_30d",
    "sessions_prev_30d",
]

print("\nNon-missing rows by observation window:")
for col in window_cols:
    print(f"{col}: {df[col].notna().sum()}")

# Basic outcome distribution
print("\nTrend-direction counts:")
print(df["trend_direction"].value_counts(dropna=False).to_string())

Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0

Missing values in fields used:
word_count                7699
trend_pct                 3388
competition               2468
search_volume             2468
main_intent               2374
scroll_rate                125
impressions_90d              0
clicks_90d                   0
days_since_last_update       0
ctr                          0
content_age_days             0
sessions_90d                 0
engagement_rate              0
avg_position                 0
trend_direction              0
content_type                 0
age_tier                     0
freshness_tier               0
impression_tier              0
position_tier                0

Non-missing rows by observation window:
impressions_90d: 30000
impressions_last_30d: 30000
impressions_prev_30d: 30000
sessions_90d: 30000
sessions_last_30d: 30000
sessions_prev_30d: 30000

Trend-direction counts:
trend_direction
down      16262
stable     5962
up         4388
new   

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Pages with fewer than 90 days of impressions:",
      (df["days_with_impressions"] < 90).sum())

print("Pages with fewer than 90 days of sessions:",
      (df["days_with_sessions"] < 90).sum())

print("Missing engagement_rate:",
      df["engagement_rate"].isna().sum())

print("Missing scroll_rate:",
      df["scroll_rate"].isna().sum())

print("\nWindow-overlap reminder:")
print("90-day totals contain the recent periods, so these fields are not independent.")


Pages with fewer than 90 days of impressions: 30000
Pages with fewer than 90 days of sessions: 29982
Missing engagement_rate: 0
Missing scroll_rate: 125

Window-overlap reminder:
90-day totals contain the recent periods, so these fields are not independent.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.